# Week 06 Lab -- Dynamic Programming

## Objectives
- Understand and implement the core DP patterns (memoization, tabulation)
- Compare the performance of naive recursion, memoization, and tabulation
- Implement LCS (Longest Common Subsequence) with DP table visualization
- Solve the 0-1 Knapsack problem with backtracking

## Lab Structure

| Exercise | Topic | Time |
|----------|-------|------|
| A-1 | Fibonacci Comparison | 10 min |
| A-2 | LCS + DP Table Visualization | 15 min |
| A-3 | 0-1 Knapsack + Backtracking | 10 min |

---
## A-1: Fibonacci Comparison (10 min)

Compare the performance of three Fibonacci implementations:
- **Naive recursion**: O(2^n) -- exponentially slow
- **Memoization** (top-down): O(n) -- cache subproblem results
- **Tabulation** (bottom-up): O(n) -- fill table iteratively

### Naive Recursion

The simplest approach: directly translate the recurrence `F(n) = F(n-1) + F(n-2)`.
This leads to an exponential number of redundant calls.

In [ ]:
import time
import sys
sys.setrecursionlimit(10000)


def fib_naive(n):
    """O(2^n): Exponential - extremely slow."""
    if n <= 1:
        return n
    return fib_naive(n - 1) + fib_naive(n - 2)


# Quick test
for i in range(10):
    print(f"fib_naive({i}) = {fib_naive(i)}")

### Memoization (Top-Down DP)

Use a dictionary to cache results of subproblems. If we have already computed `F(k)`,
we simply look it up instead of recomputing it.

In [ ]:
def fib_memo(n, memo=None):
    """O(n): Top-down with memoization."""
    if memo is None:
        memo = {}
    if n in memo:
        return memo[n]
    if n <= 1:
        return n
    memo[n] = fib_memo(n - 1, memo) + fib_memo(n - 2, memo)
    return memo[n]


# Quick test
for i in range(10):
    print(f"fib_memo({i}) = {fib_memo(i)}")

### Tabulation (Bottom-Up DP)

Build the solution from the bottom up by filling a table from `F(0)` to `F(n)`.
No recursion needed -- just a simple loop.

In [ ]:
def fib_tab(n):
    """O(n): Bottom-up tabulation."""
    if n <= 1:
        return n
    dp = [0] * (n + 1)
    dp[1] = 1
    for i in range(2, n + 1):
        dp[i] = dp[i - 1] + dp[i - 2]
    return dp[n]


# Quick test
for i in range(10):
    print(f"fib_tab({i}) = {fib_tab(i)}")

### Correctness Check

Verify that all three implementations produce the same results.

In [ ]:
# Correctness check
for i in range(20):
    assert fib_naive(i) == fib_memo(i) == fib_tab(i), f"Mismatch at n={i}"

print("All three implementations agree for n=0..19")

### Performance Comparison

Measure wall-clock time for each approach across increasing values of `n`.
Watch how the naive approach becomes unusable beyond n=35 or so.

In [ ]:
print(f"{'N':>5} | {'Naive':>12} | {'Memo':>12} | {'Tabulation':>12}")
print("-" * 50)

for n in [10, 20, 30, 35, 40]:
    # Naive (skip for large n to avoid long wait)
    if n <= 35:
        start = time.perf_counter()
        fib_naive(n)
        t_naive = time.perf_counter() - start
    else:
        t_naive = None

    start = time.perf_counter()
    fib_memo(n, {})
    t_memo = time.perf_counter() - start

    start = time.perf_counter()
    fib_tab(n)
    t_tab = time.perf_counter() - start

    naive_str = f"{t_naive:.6f}" if t_naive is not None else "too slow"
    print(f"{n:>5} | {naive_str:>12} | {t_memo:>12.6f} | {t_tab:>12.6f}")

### Discussion: Fibonacci

1. Why does the naive approach become so slow for larger `n`?
2. What is the key difference between memoization and tabulation?
3. Can you think of a way to reduce the space complexity of `fib_tab` from O(n) to O(1)?

---
## A-2: LCS + DP Table Visualization (15 min)

**Longest Common Subsequence (LCS)** finds the longest subsequence common to two strings.

**Recurrence:**
- If `X[i] == Y[j]`: `dp[i][j] = dp[i-1][j-1] + 1`
- If `X[i] != Y[j]`: `dp[i][j] = max(dp[i-1][j], dp[i][j-1])`

**Time complexity:** O(m * n)  
**Space complexity:** O(m * n)

### Building the DP Table

In [ ]:
def build_lcs_table(x, y):
    """Build the LCS DP table.

    Args:
        x: First string
        y: Second string

    Returns:
        2D DP table of size (len(x)+1) x (len(y)+1)
    """
    m, n = len(x), len(y)
    dp = [[0] * (n + 1) for _ in range(m + 1)]

    for i in range(1, m + 1):
        for j in range(1, n + 1):
            if x[i - 1] == y[j - 1]:
                dp[i][j] = dp[i - 1][j - 1] + 1
            else:
                dp[i][j] = max(dp[i - 1][j], dp[i][j - 1])

    return dp


# Test
x, y = "ABCBDAB", "BDCAB"
dp = build_lcs_table(x, y)
print(f"LCS length of '{x}' and '{y}': {dp[len(x)][len(y)]}")

### Backtracking to Recover the LCS

Starting from `dp[m][n]`, trace back to `dp[0][0]`:
- If characters match: move diagonally (include in LCS)
- Otherwise: move in the direction of the larger value

In [ ]:
def backtrack_lcs(dp, x, y):
    """Backtrack through the DP table to recover the actual LCS.

    Args:
        dp: LCS DP table
        x: First string
        y: Second string

    Returns:
        (LCS string, backtrack path list)
        Path: [(i, j, action), ...] where action = 'match'|'up'|'left'
    """
    lcs = []
    path = []
    i, j = len(x), len(y)

    while i > 0 and j > 0:
        if x[i - 1] == y[j - 1]:
            lcs.append(x[i - 1])
            path.append((i, j, "match"))
            i -= 1
            j -= 1
        elif dp[i - 1][j] >= dp[i][j - 1]:
            path.append((i, j, "up"))
            i -= 1
        else:
            path.append((i, j, "left"))
            j -= 1

    lcs.reverse()
    path.reverse()

    return "".join(lcs), path


# Test
lcs_str, path = backtrack_lcs(dp, x, y)
print(f"LCS: '{lcs_str}'")
print(f"Path: {path}")

### Visualizing the DP Table

Print the DP table with backtrack path highlighted:
- `[n]` = match (character included in LCS)
- `(n)` = backtrack path (not a match)

In [ ]:
def print_dp_table(dp, x, y, path=None):
    """Print the DP table with optional path highlighting."""
    m, n = len(x), len(y)

    # Convert path to sets for quick lookup
    match_cells = set()
    path_cells = set()
    if path:
        for i, j, action in path:
            path_cells.add((i, j))
            if action == "match":
                match_cells.add((i, j))

    cell_width = 4

    # Y character header
    print(f"    {'':>{cell_width}}    ", end="")
    for j in range(n + 1):
        if j == 0:
            print(f"{'':>{cell_width}}", end=" ")
        else:
            print(f"{y[j-1]:>{cell_width}}", end=" ")
    print()

    # Index header
    print(f"    {'':>{cell_width}}    ", end="")
    for j in range(n + 1):
        print(f"{j:>{cell_width}}", end=" ")
    print()

    print(f"    {'':>{cell_width}}   {'---' * (n + 1) * 2}")

    # Table body
    for i in range(m + 1):
        if i == 0:
            row_label = " "
        else:
            row_label = x[i - 1]

        print(f"    {row_label:>{cell_width}} {i} |", end="")

        for j in range(n + 1):
            val = dp[i][j]
            if (i, j) in match_cells:
                print(f"[{val:>{cell_width - 2}}]", end=" ")
            elif (i, j) in path_cells:
                print(f"({val:>{cell_width - 2}})", end=" ")
            else:
                print(f"{val:>{cell_width}}", end=" ")
        print()

    print()

In [ ]:
def lcs_analysis(x, y, label=""):
    """Perform full LCS analysis and display results."""
    if label:
        print(f"\n{'='*60}")
        print(f"[{label}]")
        print(f"{'='*60}")

    print(f"\n  X = \"{x}\"")
    print(f"  Y = \"{y}\"")

    # Build DP table
    dp = build_lcs_table(x, y)

    # Backtrack
    lcs_str, path = backtrack_lcs(dp, x, y)

    # Print DP table
    print(f"\n  --- DP Table ---")
    print(f"  [n] = match (in LCS), (n) = backtrack path")
    print_dp_table(dp, x, y, path)

    # Print results
    lcs_length = dp[len(x)][len(y)]
    print(f"  LCS length: {lcs_length}")
    print(f"  LCS: \"{lcs_str}\"")

    # Backtrack path visualization
    print(f"\n  --- Backtrack Path ---")
    for i, j, action in path:
        if action == "match":
            print(f"    ({i},{j}): X[{i}]=Y[{j}]='{x[i-1]}' -> diagonal (match!)")
        elif action == "up":
            print(f"    ({i},{j}): X[{i}]='{x[i-1]}' != Y[{j}]='{y[j-1]}' -> move up")
        else:
            print(f"    ({i},{j}): X[{i}]='{x[i-1]}' != Y[{j}]='{y[j-1]}' -> move left")

    # Show LCS positions in original strings
    print(f"\n  --- LCS Position Markers ---")
    x_marks = [" "] * len(x)
    y_marks = [" "] * len(y)

    for i, j, action in path:
        if action == "match":
            x_marks[i - 1] = "^"
            y_marks[j - 1] = "^"

    print(f"  X: {' '.join(x)}")
    print(f"     {' '.join(x_marks)}")
    print(f"  Y: {' '.join(y)}")
    print(f"     {' '.join(y_marks)}")

    return lcs_str, lcs_length

### Example 1: Textbook Example

In [ ]:
lcs_analysis("ABCBDAB", "BDCAB", "Example 1: ABCBDAB vs BDCAB")

### Example 2: Another Classic

In [ ]:
lcs_analysis("AGGTAB", "GXTXAYB", "Example 2: AGGTAB vs GXTXAYB")

### Example 3: Identical Strings

In [ ]:
lcs_analysis("ABC", "ABC", "Example 3: Identical strings ABC vs ABC")

### Example 4: No Common Subsequence

In [ ]:
lcs_analysis("ABC", "XYZ", "Example 4: No common chars ABC vs XYZ")

### Exercise: Try Your Own Strings

Modify the strings below and run the cell to see the LCS analysis.

In [ ]:
# TODO: Try your own strings
my_x = "ALGORITHM"
my_y = "ALTRUISTIC"

lcs_analysis(my_x, my_y, f"Your example: {my_x} vs {my_y}")

### Discussion: LCS

1. The LCS algorithm has time complexity O(m * n). Why can't we do better with a simple approach?
2. How is LCS used in real-world applications like `git diff` or DNA sequence alignment?
3. What would happen if we wanted to find all possible LCS strings (not just one)?

### LCS Summary

| Property | Value |
|----------|-------|
| Time complexity | O(m * n) |
| Space complexity | O(m * n) for the DP table |
| Recurrence (match) | `dp[i][j] = dp[i-1][j-1] + 1` |
| Recurrence (no match) | `dp[i][j] = max(dp[i-1][j], dp[i][j-1])` |

**Applications:** Text diff (git diff), DNA sequence comparison, Edit Distance

---
## A-3: 0-1 Knapsack + Backtracking (10 min)

Given a set of items, each with a value and weight, determine which items to include
in a knapsack of limited capacity to maximize total value.

**Key constraint:** Each item can either be taken or left (0-1), not fractionally divided.

### Implementation with Backtracking

In [ ]:
def knapsack_01(capacity, items):
    """
    Solve the 0-1 knapsack problem.

    Args:
        capacity: Maximum weight the knapsack can hold
        items: list of (value, weight, name)

    Returns:
        (max_value, selected_items)
    """
    n = len(items)
    dp = [[0] * (capacity + 1) for _ in range(n + 1)]

    for i in range(1, n + 1):
        v, w, _ = items[i - 1]
        for j in range(capacity + 1):
            dp[i][j] = dp[i - 1][j]
            if w <= j and dp[i - 1][j - w] + v > dp[i][j]:
                dp[i][j] = dp[i - 1][j - w] + v

    # Backtrack to find selected items
    selected = []
    j = capacity
    for i in range(n, 0, -1):
        if dp[i][j] != dp[i - 1][j]:
            selected.append(items[i - 1])
            j -= items[i - 1][1]

    return dp[n][capacity], list(reversed(selected))

### Running the Knapsack Example

In [ ]:
items = [
    (60, 10, "Laptop"),
    (100, 20, "Camera"),
    (120, 30, "Painting"),
    (40, 5, "Book"),
]
capacity = 50

print(f"Items: {[(name, f'v={v}, w={w}') for v, w, name in items]}")
print(f"Capacity: {capacity}\n")

max_val, selected = knapsack_01(capacity, items)
print(f"Maximum value: {max_val}")
print(f"Selected items:")
total_weight = 0
for v, w, name in selected:
    print(f"  {name}: value={v}, weight={w}")
    total_weight += w
print(f"Total weight: {total_weight}/{capacity}")

### Exercise: Visualize the Knapsack DP Table

Complete the function below to print the DP table for the knapsack problem.
Rows represent items (0 = no items, 1 = first item, ...), columns represent capacities (0 to W).

In [ ]:
def knapsack_with_table(capacity, items):
    """Solve knapsack and print the DP table."""
    n = len(items)
    dp = [[0] * (capacity + 1) for _ in range(n + 1)]

    for i in range(1, n + 1):
        v, w, _ = items[i - 1]
        for j in range(capacity + 1):
            dp[i][j] = dp[i - 1][j]
            if w <= j and dp[i - 1][j - w] + v > dp[i][j]:
                dp[i][j] = dp[i - 1][j - w] + v

    # TODO: Print the DP table
    # Hint: Use a compact format since the table can be wide.
    # Show every 5th column to keep it readable, e.g., columns 0, 5, 10, ...
    step = 5
    cols = list(range(0, capacity + 1, step))
    if capacity not in cols:
        cols.append(capacity)

    # Header
    header = f"{'Item':>12} |" + "".join(f" w={c:>2} " for c in cols)
    print(header)
    print("-" * len(header))

    for i in range(n + 1):
        if i == 0:
            label = "(none)"
        else:
            _, _, name = items[i - 1]
            label = name
        row = f"{label:>12} |" + "".join(f" {dp[i][c]:>4} " for c in cols)
        print(row)

    print(f"\nMaximum value: {dp[n][capacity]}")
    return dp


# Run with table visualization
knapsack_with_table(capacity, items)

### Exercise: Try Different Items

Modify the items and capacity below to test different scenarios.

In [ ]:
# TODO: Try your own items and capacity
my_items = [
    (10, 5, "Item A"),
    (40, 4, "Item B"),
    (30, 6, "Item C"),
    (50, 3, "Item D"),
]
my_capacity = 10

max_val, selected = knapsack_01(my_capacity, my_items)
print(f"Maximum value: {max_val}")
print(f"Selected items:")
total_weight = 0
for v, w, name in selected:
    print(f"  {name}: value={v}, weight={w}")
    total_weight += w
print(f"Total weight: {total_weight}/{my_capacity}")

### Discussion: 0-1 Knapsack

1. Why can't we use a greedy approach (e.g., pick by highest value/weight ratio) for the 0-1 knapsack?
2. What is the time and space complexity of this DP solution?
3. How does the **fractional knapsack** (where items can be split) differ in approach?

---
## Summary

| Algorithm | Time | Space | Key Idea |
|-----------|------|-------|----------|
| Fibonacci (naive) | O(2^n) | O(n) stack | Exponential redundant calls |
| Fibonacci (memo) | O(n) | O(n) | Cache subproblem results (top-down) |
| Fibonacci (tab) | O(n) | O(n) or O(1) | Fill table iteratively (bottom-up) |
| LCS | O(m*n) | O(m*n) | 2D DP table + backtracking |
| 0-1 Knapsack | O(n*W) | O(n*W) | Take-or-skip decision at each item |

### DP Recipe
1. **Define subproblems** -- What does `dp[i]` (or `dp[i][j]`) represent?
2. **Write the recurrence** -- How does `dp[i]` relate to smaller subproblems?
3. **Identify base cases** -- What are the trivial cases?
4. **Determine computation order** -- Bottom-up or top-down?
5. **Backtrack if needed** -- Recover the actual solution, not just the optimal value.